In [2]:
import json
import os
import pandas as pd

In [ ]:
# List of your local manually downloaded JSON files
json_files = [
    
    "../data/raw_fire_data011-015.json",
    
]
all_features = []

output_file = "../data/boku_waldbrand_02-11-2026(teil_011-015).csv"

print("Reading and parsing manual GeoJSON chunk files...")
print("-" * 50)

# 1. Loop through and load the contents of each file
for file_name in json_files:
    if os.path.exists(file_name):
        try:
            with open(file_name, "r", encoding="utf-8") as f:
                data = json.load(f)
                # Isolate the core array list containing the forest fire nodes
                features = data.get("features", [])
                print(f"{file_name}: Loaded {len(features)} records.")
                all_features.extend(features)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
    else:
        print(f"Warning: File '{file_name}' not found in the directory.")

if not all_features:
    print("\nCritical Error: No records could be processed. Please check file locations.")
    exit()

print("-" * 50)
print(f"Total raw features gathered in memory: {len(all_features)}")
print("Flattening nested GeoJSON structures into tabular layout...")

# 2. Extract properties and geographical coordinates from the GeoJSON objects
parsed_records = []
for feature in all_features:
    props = feature.get("properties", {})
    geometry = feature.get("geometry", {})
    coordinates = geometry.get("coordinates", [None, None])

    # Unpack native coordinate point elements explicitly into clean table columns
    if coordinates and len(coordinates) == 2:
        props["longitude"] = coordinates[0]
        props["latitude"] = coordinates[1]
    else:
        props["longitude"] = None
        props["latitude"] = None

    parsed_records.append(props)

# 3. Load into structured Pandas DataFrame matrix
df = pd.DataFrame(parsed_records)

# 4. Drop any overlapping duplicate records on edge bounds using the unique fire key
initial_count = len(df)
df = df.drop_duplicates(subset=["waldbrandid"])
final_count = len(df)

if initial_count != final_count:
    print(f"Removed {initial_count - final_count} overlapping duplicate records.")

# 5. Export clean table locally to an Excel/GIS compatible CSV file
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\n" + "=" * 65)
print("DATA CONVERSION COMPLETED SUCCESSFULLY")
print(f"Pandas DataFrame Rows:      {final_count}")
print(f"Local clean CSV file saved:  {output_file}")
print("=" * 65)

# Preview the structure of the data fields inside the active dataframe
print("\nSample Preview of the Dataframe Matrix (First 5 Rows):")
print(df[["waldbrandid", "jahr", "bundesland", "gemeinde_neu", "brandflaeche"]].head())


Reading and parsing manual GeoJSON chunk files...
--------------------------------------------------
../data/raw_fire_data011-015.json: Loaded 1746 records.
--------------------------------------------------
Total raw features gathered in memory: 1746
Flattening nested GeoJSON structures into tabular layout...

DATA CONVERSION COMPLETED SUCCESSFULLY
Pandas DataFrame Rows:      1746
Local clean CSV file saved:  boku_waldbrand_02-11-2026(teil_011-015).csv

Sample Preview of the Dataframe Matrix (First 5 Rows):
   waldbrandid  jahr bundesland                gemeinde_neu  brandflaeche
0   2015120702  2015          K                     Reißeck           500
1   2011071101  2011          K  Techelsberg am Wörther See           200
2   2011033001  2011          T          Kirchdorf in Tirol         14670
3   2014122401  2014          T                     Serfaus            50
4   2013080101  2013        NOE     St. Egyden am Steinfeld        199700
